# Financial RAG Pipeline — Apple Inc. 10-K Analysis

This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) pipeline over Apple's SEC 10-K filings. It compares two chunking strategies (fixed-size vs. semantic), adds a reranking step, and evaluates all three configurations using RAGAS metrics.

**Tech Stack:** LangChain · Pinecone · HuggingFace (BAAI/bge-base-en-v1.5) · OpenAI (gpt-4o-mini) · TypeSafe AI Jev · RAGAS

**Documents:** Apple Inc. 10-K filings (FY2024, FY2025) via SEC EDGAR


In [3]:
from IPython.display import display, HTML

try:
    with open("/content/financial_rag_pipeline_architecture.html", "r") as file:
        html_content = file.read()
    display(HTML(html_content))
except FileNotFoundError:
    print("ℹ️ Architecture diagram not found. Upload financial_rag_pipeline_architecture.html to /content/ to display it.")

## 1. Setup

### 1.1 Install Dependencies

> ⚠️ Do NOT install `torch` here — Colab already ships with a CUDA-enabled build. Installing via pip overwrites it with a CPU-only version.


In [5]:
# Do NOT install torch here — Colab already has a CUDA-enabled version
# Installing it via pip overwrites it with a CPU build

!pip install -q langchain langchain-community langchain-huggingface langchain-groq
!pip install -q langchain-pinecone pinecone
!pip install -q sentence-transformers
!pip install -q sec-edgar-downloader
!pip install -q pypdf unstructured
!pip install -q ragas datasets
!pip install -q typesafe-sdk
!pip install -q langchain-experimental
!pip install -q langchain-openai
!pip install -q -U ragas langchain-community langchain-openai


### 1.2 Verify GPU

Confirm CUDA is available before loading the embedding model. Switch runtime to **T4 GPU** in Runtime → Change runtime type if needed.


In [6]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
!nvidia-smi | head -20

PyTorch: 2.11.0+cu130
CUDA: True
Wed Oct  7 13:56:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             15W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------

### 1.3 API Keys

Keys are loaded from Colab Secrets (the 🔑 icon in the sidebar). Required: `PINECONE_API_KEY`, `OPENAI_API_KEY`, `HUGGINGFACEHUB_API_TOKEN`.


In [7]:
import os
from google.colab import userdata

# Paste your keys when prompted — they won't show on screen
os.environ["PINECONE_API_KEY"]    = userdata.get('PINECONE_API_KEY')
os.environ["OPENAI_API_KEY"]   = userdata.get('OPENAI_API_KEY')
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get('HUGGINGFACEHUB_API_TOKEN')

print("✅ Keys set")

✅ Keys set


### 1.4 Test Connections


Test OpenAI Connection


In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

response = llm.invoke("Say 'OpenAI is working' in exactly 3 words.")
print(response.content)

OpenAI is functioning.


Test Pinecone connection


In [9]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

INDEX_NAME = "financial-rag"

# Create the index if it doesn't exist yet
if INDEX_NAME not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,          # matches BAAI/bge-base-en-v1.5
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")  # free tier
    )
    print(f"✅ Created index: {INDEX_NAME}")
else:
    print(f"✅ Index already exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())

✅ Index already exists: financial-rag
{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'fixed': {'vector_count': 12142},
                'semantic': {'vector_count': 2529}},
 'total_vector_count': 14671,
 'vector_type': 'dense'}


Test HuggingFace embedding model


In [10]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# Auto-detect — uses GPU if available, falls back to CPU cleanly
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

test_vector = embeddings.embed_query("What is Apple's net revenue?")
print(f"✅ Embedding model loaded. Vector dimension: {len(test_vector)}")

Using device: cuda
PyTorch version: 2.11.0+cu130
CUDA available: True


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded. Vector dimension: 768


## 2. Data Ingestion

### 2.1 Download SEC Filings

We download Apple's last 2 annual reports (10-K) directly from SEC EDGAR using `sec-edgar-downloader`. No API key required — SEC only asks for a name and email for logging purposes.


In [11]:
from sec_edgar_downloader import Downloader

# SEC requires a name and email (any valid email, it's just for their logs)
dl = Downloader(company_name="LearningProject", email_address="test@example.com")

# Download Apple's last 2 annual reports (10-K filings)
# They save to ./sec-edgar-filings/AAPL/10-K/
dl.get("10-K", "AAPL", limit=2)

print("✅ Download complete")

✅ Download complete


### 2.2 Find Downloaded Files


In [12]:
import os

# Walk through the downloaded folder and collect all .htm / .txt files
filing_files = []

for root, dirs, files in os.walk("sec-edgar-filings"):
    for file in files:
        if file.endswith(".htm") or file.endswith(".txt"):
            full_path = os.path.join(root, file)
            filing_files.append(full_path)
            print(full_path)

print(f"\nTotal files found: {len(filing_files)}")

sec-edgar-filings/AAPL/10-K/0000320193-24-000123/full-submission.txt
sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt

Total files found: 2


### 2.3 Load Documents

Each filing is a single large `.txt` file containing the full SEC submission. We tag each document with its source path as metadata for traceability.


In [13]:
from langchain_community.document_loaders import TextLoader

documents = []

for path in filing_files:
    try:
        loader = TextLoader(path, encoding="utf-8", autodetect_encoding=True)
        docs = loader.load()
        # Tag each doc with which filing it came from
        for doc in docs:
            doc.metadata["source"] = path
        documents.extend(docs)
        print(f"✅ Loaded: {path}  →  {len(docs)} doc(s)")
    except Exception as e:
        print(f"⚠️  Skipped: {path}  →  {e}")

print(f"\nTotal documents loaded: {len(documents)}")

✅ Loaded: sec-edgar-filings/AAPL/10-K/0000320193-24-000123/full-submission.txt  →  1 doc(s)
✅ Loaded: sec-edgar-filings/AAPL/10-K/0000320193-25-000079/full-submission.txt  →  1 doc(s)

Total documents loaded: 2


/tmp/ipykernel_1985/2542833511.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 3. Text Preprocessing

The raw EDGAR files contain SGML tags, HTML entities (`&#160;`, `&#9746;`), XBRL data blocks, and filing header metadata. We apply two cleaning steps:
1. **`clean_edgar_text`** — strips tags, decodes HTML entities, collapses whitespace
2. **`remove_edgar_header`** — skips everything before "UNITED STATES" (the actual 10-K cover page)


In [14]:
import html
import re

def clean_edgar_text(text):
    # Decode HTML entities first: &#160; → space, &#9746; → ✖ etc.
    text = html.unescape(text)

    # Remove remaining HTML/SGML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove XBRL artifacts
    text = re.sub(r'ix:[^\s]+', ' ', text)

    # Remove lines that are just symbols, dashes, or whitespace
    text = re.sub(r'^[\-_=\s\u00a0]+$', '', text, flags=re.MULTILINE)

    # Collapse multiple blank lines and spaces
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {3,}', ' ', text)

    return text.strip()

def remove_edgar_header(text):
    marker = "UNITED STATES"
    pos = text.find(marker)

    if pos != -1:
        return text[pos:]

    return text

# Re-apply to original documents
for doc in documents:
    doc.page_content = clean_edgar_text(doc.page_content)
    doc.page_content = remove_edgar_header(doc.page_content)

print(documents[0].page_content[:500])

UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM  10-K (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended September 28 , 2024 or ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from              to               . Commission File Number: 001-36743 Apple Inc. (Exact name of Registrant as specified in its charter) California 94-24


## 4. Chunking Strategies

Chunking determines how documents are split before embedding. We compare two strategies stored in separate Pinecone namespaces.

### 4A. Fixed-Size Chunking

Splits every 512 characters with 64-character overlap using `RecursiveCharacterTextSplitter`. Fast and simple, but cuts mid-sentence and mid-table — a known weakness for financial documents with dense tables.


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)

fixed_chunks_raw = fixed_splitter.split_documents(documents)

# Filter out noisy chunks
def is_clean_chunk(chunk):
    text = chunk.page_content.strip()
    if len(text) < 100:                          # too short to be useful
        return False
    alpha_ratio = sum(c.isalpha() for c in text) / len(text)
    if alpha_ratio < 0.4:                        # mostly numbers/symbols/whitespace
        return False
    return True

fixed_chunks = [c for c in fixed_chunks_raw if is_clean_chunk(c)]

print(f"Raw chunks:      {len(fixed_chunks_raw)}")
print(f"After filtering: {len(fixed_chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in fixed_chunks) // len(fixed_chunks)} chars")

print("\n--- Chunk 5 ---")
print(fixed_chunks[5].page_content)
print("\n--- Chunk 6 ---")
print(fixed_chunks[6].page_content)

Raw chunks:      14119
After filtering: 12142
Avg chunk length: 431 chars

--- Chunk 5 ---
. Yes     ☒       No    ☐ Indicate by check mark if the Registrant is not required to file reports pursuant to Section 13 or Section 15(d) of the Act

--- Chunk 6 ---
. Yes    ☐        No     ☒ Indicate by check mark whether the Registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of the Securities Exchange Act of 1934 during the preceding 12 months (or for such shorter period that the Registrant was required to file such reports), and (2) has been subject to such filing requirements for the past 90 days


Clear any partial data


In [16]:
# Clear the namespace first — avoids duplicates from the interrupted run
from pinecone.exceptions import NotFoundException

print("Clearing any partial data in namespace='fixed'...")

try:
    index.delete(delete_all=True, namespace="fixed")
    print("Cleared")
except NotFoundException:
    print("Namespace 'fixed' doesn't exist yet — nothing to clear.")

Clearing any partial data in namespace='fixed'...
Cleared


### 4A continued: Embed & Upload to Pinecone (`namespace='fixed'`)

Chunks are embedded with `BAAI/bge-base-en-v1.5` (768-dim) and uploaded in batches of 100.


In [17]:
from tqdm import tqdm
from langchain_pinecone import PineconeVectorStore
BATCH_SIZE = 100

print(f"Uploading {len(fixed_chunks)} chunks in batches of {BATCH_SIZE}...")
print("Estimated time: ~3 min on GPU, ~30-40 min on CPU\n")

# First batch initializes the vectorstore
fixed_vectorstore = PineconeVectorStore.from_documents(
    documents=fixed_chunks[:BATCH_SIZE],
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace="fixed"
)

# Rest of the batches
for i in tqdm(range(BATCH_SIZE, len(fixed_chunks), BATCH_SIZE)):
    batch = fixed_chunks[i : i + BATCH_SIZE]
    fixed_vectorstore.add_documents(batch)

print(f"\n✅ Upload complete")
stats = index.describe_index_stats()
print(f"Vectors in 'fixed' namespace: {stats['namespaces'].get('fixed', {}).get('vector_count', 0)}")

Uploading 12142 chunks in batches of 100...
Estimated time: ~3 min on GPU, ~30-40 min on CPU



100%|██████████| 121/121 [04:22<00:00,  2.17s/it]



✅ Upload complete
Vectors in 'fixed' namespace: 12142


### 4B. Semantic Chunking

Uses `SemanticChunker` which embeds each sentence and cuts where cosine distance between adjacent sentences spikes above the 65th percentile. This keeps topically coherent content together — critical for financial tables and multi-paragraph analyses.

**Threshold tuning:** 95 → 13,050 avg chars (too large) · 80 → 3,656 · 70 → 2,568 · **65 → 2,246** ✅


In [18]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=65
)

print("Running semantic chunking...")
semantic_chunks_raw = semantic_splitter.split_documents(documents)

semantic_chunks = [c for c in semantic_chunks_raw if is_clean_chunk(c)]

print(f"Raw semantic chunks:      {len(semantic_chunks_raw)}")
print(f"After filtering:          {len(semantic_chunks)}")
print(f"Avg chunk length: {sum(len(c.page_content) for c in semantic_chunks) // len(semantic_chunks)} chars")

/tmp/ipykernel_1985/2511489152.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Running semantic chunking...
Raw semantic chunks:      3080
After filtering:          2245
Avg chunk length: 2246 chars


### 4B continued: Handle Oversized Chunks & Upload to Pinecone (`namespace='semantic'`)

Pinecone has a 40KB metadata limit per vector. Some semantic chunks exceed this (outlier tables can be 20,000+ chars). We apply a secondary split at 8,000 chars before uploading.


In [19]:
from langchain_pinecone import PineconeVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

# Fix: split any chunk that exceeds Pinecone's 40KB metadata limit
MAX_CHARS = 8000
overflow_splitter = RecursiveCharacterTextSplitter(chunk_size=MAX_CHARS, chunk_overlap=100)

upload_chunks = []
for chunk in semantic_chunks:
    if len(chunk.page_content) > MAX_CHARS:
        upload_chunks.extend(overflow_splitter.split_documents([chunk]))
    else:
        upload_chunks.append(chunk)

print(f"Semantic chunks before: {len(semantic_chunks)}")
print(f"Semantic chunks after:  {len(upload_chunks)}")

# Clear namespace first
print("\nClearing namespace='semantic'...")
try:
    index.delete(delete_all=True, namespace="semantic")
    print("Cleared.")
except Exception as e:
    print("Already empty, skipping.")

BATCH_SIZE = 100
print(f"\nUploading {len(upload_chunks)} chunks to namespace='semantic'...\n")

semantic_vectorstore = PineconeVectorStore.from_documents(
    documents=upload_chunks[:BATCH_SIZE],
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace="semantic"
)

for i in tqdm(range(BATCH_SIZE, len(upload_chunks), BATCH_SIZE)):
    batch = upload_chunks[i : i + BATCH_SIZE]
    semantic_vectorstore.add_documents(batch)

print(f"\n✅ Upload complete")
stats = index.describe_index_stats()
print(f"Vectors in 'fixed':    {stats['namespaces'].get('fixed', {}).get('vector_count', 0)}")
print(f"Vectors in 'semantic': {stats['namespaces'].get('semantic', {}).get('vector_count', 0)}")

Semantic chunks before: 2245
Semantic chunks after:  2529

Clearing namespace='semantic'...
Cleared.

Uploading 2529 chunks to namespace='semantic'...



100%|██████████| 25/25 [01:29<00:00,  3.59s/it]



✅ Upload complete
Vectors in 'fixed':    12142
Vectors in 'semantic': 2529


## 5. Retrieval Testing

Connect to both Pinecone namespaces and run the same queries through both strategies to visually compare retrieval quality before formal evaluation.

### 5.1 Connect to Vector Stores


In [20]:
from langchain_pinecone import PineconeVectorStore

# Connect to existing namespaces — no re-uploading needed
fixed_vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="fixed"
)

semantic_vectorstore = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="semantic"
)

print("✅ Connected to both namespaces")

✅ Connected to both namespaces


### 5.2 Test Queries

A mix of specific numerical questions and broad qualitative questions — the same set used to sanity-check retrieval before running RAGAS.


In [21]:
# These are the kinds of questions a financial analyst would ask
# Mix of specific (numbers) and broad (strategy/risk) questions
test_queries = [
    "What was Apple's total net revenue in fiscal year 2024?",
    "What are the main risk factors related to Apple's supply chain?",
    "How much did Apple spend on research and development?",
    "What is Apple's strategy for services revenue growth?",
    "What were Apple's earnings per share in 2024?",
]

### 5.3 Side-by-Side Comparison: Fixed vs Semantic

For each query, we show the top-1 chunk from each strategy. Key observation: fixed-size chunks cut financial tables mid-row, making numbers meaningless without their headers. Semantic chunks keep tables intact.


In [22]:
TOP_K = 3  # retrieve top 3 chunks per query

for query in test_queries:
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    fixed_results   = fixed_vectorstore.similarity_search(query, k=TOP_K)
    semantic_results = semantic_vectorstore.similarity_search(query, k=TOP_K)

    print(f"\n--- FIXED (top 1 of {TOP_K}) ---")
    print(fixed_results[0].page_content[:600])
    print(f"[length: {len(fixed_results[0].page_content)} chars]")

    print(f"\n--- SEMANTIC (top 1 of {TOP_K}) ---")
    print(semantic_results[0].page_content[:600])
    print(f"[length: {len(semantic_results[0].page_content)} chars]")
    print()

QUERY: What was Apple's total net revenue in fiscal year 2024?

--- FIXED (top 1 of 3) ---
. Total net sales include $ 7.7  billion of revenue recognized in 2024 that was included in deferred revenue as of September 30, 2023, $ 8.2  billion of revenue recognized in 2023 that was included in deferred revenue as of September 24, 2022, and $ 7.5  billion of revenue recognized in 2022 that was included in deferred revenue as of September 25, 2021. Apple Inc
[length: 366 chars]

--- SEMANTIC (top 1 of 3) ---
Apple Inc. | 2024 Form 10-K | 22 Products and Services Performance The following table shows net sales by category for 2024, 2023 and 2022 (dollars in millions): 2024 Change 2023 Change 2022 iPhone $ 201,183  —  % $ 200,583  (2) % $ 205,489  Mac 29,984  2  % 29,357  (27) % 40,177  iPad 26,694  (6) % 28,300  (3) % 29,292  Wearables, Home and Accessories 37,005  (7) % 39,845  (3) % 41,241  Services (1) 96,169  13  % 85,200  9  % 78,129  Total net sales $ 391,035  2  % $ 383,285  (3) % $ 3

## 6. Reranking

A bi-encoder (Pinecone similarity search) retrieves a wider set of semantically similar chunks. **Jev** is then used as the reranker: it sees the query together with the retrieved candidates and makes one narrow relevance decision for each candidate.

The implementation follows TypeSafe's System One pattern: Python keeps control of retrieval and sorting, while Jev handles the bounded judgment. All candidate relevance questions are sent together in one `system_one(...)` request, and the returned `Noul` probabilities are used as reranking scores.

**Strategy:** Retrieve top-10 candidates with bi-encoder → Jev scores candidate relevance → sort by `P(relevant)` → return top-3.

### 6.1 Load Jev Reranker


In [23]:
from typesafe_sdk import Noul, NoulCriteria, TypeSafeClient
from google.colab import userdata

# Add TYPESAFE_API_KEY in Colab → Secrets before running this cell.
# TypeSafeClient reads this environment variable when system_one(...) is called.
TYPESAFE_API_KEY = userdata.get("TYPESAFE_API_KEY")
if not TYPESAFE_API_KEY:
    raise ValueError(
        "TYPESAFE_API_KEY not found. Add it in Colab → Secrets and rerun this cell."
    )

os.environ["TYPESAFE_API_KEY"] = TYPESAFE_API_KEY

print("TypeSafe AI API key loaded")
print("Jev reranker ready")


TypeSafe AI API key loaded
Jev reranker ready


### 6.2 Define Retrieval Functions


In [24]:
def retrieve_no_rerank(query, k=10):
    """Standard vector similarity search"""
    results = semantic_vectorstore.similarity_search(query, k=k)
    return results

def retrieve_with_rerank(query, k=10, top_n=3):
    """Vector search → Jev re-scores → returns best top_n"""
    # Step 1: cast a wide net — get more candidates than we need
    candidates = semantic_vectorstore.similarity_search(query, k=k)

    # Step 2: give Jev only the state needed for the relevance decision
    state = {
        "query": query,
        "candidates": [
            {
                "id": i,
                "text": c.page_content
            }
            for i, c in enumerate(candidates)
        ]
    }

    # One narrow Noul question per candidate.
    # TypeSafe evaluates the independent questions together in one System One request.
    questions = {
        f"candidate_{i}": Noul(
            instructions=(
                f"Is candidate {i} directly relevant and useful for answering the query? "
                "Judge whether it contains evidence that helps answer the question, "
                "not just similar financial words."
            ),
            criteria=NoulCriteria(
                true=(
                    "The candidate contains information that directly helps answer the query, "
                    "including the correct metric, fiscal period, fact, explanation, or risk."
                ),
                false=(
                    "The candidate is unrelated, only loosely similar, refers to the wrong "
                    "fact or period, or does not help answer the query."
                ),
            ),
        )
        for i in range(len(candidates))
    }

    # TypeSafe's documented System One pattern:
    # state + typed questions -> typed answers
    with TypeSafeClient() as client:
        response = client.system_one(
            state=state,
            questions=questions,
        )

    # Step 3: use P(yes) from each Noul as the reranking score
    results = [
        {
            "id": i,
            "score": response.answers[f"candidate_{i}"].noul,
        }
        for i in range(len(candidates))
    ]
    results.sort(key=lambda x: x["score"], reverse=True)

    # Step 4: map scores back to original documents
    reranked_docs = []
    for r in results[:top_n]:
        doc = candidates[r["id"]]
        doc.metadata["rerank_score"] = r["score"]
        reranked_docs.append(doc)

    return reranked_docs

print("✅ Retrieval functions ready")


✅ Retrieval functions ready


### 6.3 Fixed-Size Retrieval Function


In [25]:
def retrieve_fixed(query, k=3):
    """Vector similarity search on fixed-size namespace"""
    results = fixed_vectorstore.similarity_search(query, k=k)
    return results

print("✅ Fixed retrieval function ready")

✅ Fixed retrieval function ready


### 6.4 Compare: Semantic With vs Without Reranking

Same 3 queries run through semantic retrieval with and without the reranker. The reranker's confidence score is shown alongside the result.


In [26]:
compare_queries = [
    "What was Apple's total net revenue in fiscal year 2024?",
    "What were Apple's earnings per share in 2024?",
    "How much did Apple spend on research and development?",
]

for query in compare_queries:
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    no_rerank  = retrieve_no_rerank(query, k=10)
    reranked   = retrieve_with_rerank(query, k=10, top_n=3)

    print("\n--- WITHOUT RERANKING (top 1) ---")
    print(no_rerank[0].page_content[:600])

    print("\n--- WITH RERANKING (top 1) ---")
    print(reranked[0].page_content[:600])
    print(f"Rerank score: {reranked[0].metadata.get('rerank_score', 'N/A')}")
    print()

QUERY: What was Apple's total net revenue in fiscal year 2024?

--- WITHOUT RERANKING (top 1) ---
Apple Inc. | 2024 Form 10-K | 22 Products and Services Performance The following table shows net sales by category for 2024, 2023 and 2022 (dollars in millions): 2024 Change 2023 Change 2022 iPhone $ 201,183  —  % $ 200,583  (2) % $ 205,489  Mac 29,984  2  % 29,357  (27) % 40,177  iPad 26,694  (6) % 28,300  (3) % 29,292  Wearables, Home and Accessories 37,005  (7) % 39,845  (3) % 41,241  Services (1) 96,169  13  % 85,200  9  % 78,129  Total net sales $ 391,035  2  % $ 383,285  (3) % $ 394,328  (1)  Services net sales include amortization of the deferred value of services bundled in the sales p

--- WITH RERANKING (top 1) ---
Apple Inc. | 2024 Form 10-K | 22 Products and Services Performance The following table shows net sales by category for 2024, 2023 and 2022 (dollars in millions): 2024 Change 2023 Change 2022 iPhone $ 201,183  —  % $ 200,583  (2) % $ 205,489  Mac 29,984  2  % 29,357  (2

## 7. Answer Generation

### 7.1 Build RAG Chain

Full pipeline: retrieve → (optionally rerank) → format context → generate with `gpt-4o-mini`. The prompt instructs the model to answer **only** from the provided context to prevent hallucination.


In [27]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt_template = ChatPromptTemplate.from_template("""
You are a financial analyst assistant. Answer the question using ONLY the context provided below.
If the context does not contain enough information to answer, say "I don't have enough context to answer this."
Always cite specific numbers and figures when available.

Context:
{context}

Question: {question}

Answer:
""")

def format_context(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

def rag_answer(query, use_reranker=True, k=10, top_n=3):
    if use_reranker:
        docs = retrieve_with_rerank(query, k=k, top_n=top_n)
    else:
        docs = retrieve_no_rerank(query, k=top_n)
    context = format_context(docs)
    chain   = prompt_template | llm | StrOutputParser()
    answer  = chain.invoke({"context": context, "question": query})
    return {"answer": answer, "sources": docs}

print("RAG chain ready with OpenAI gpt-4o-mini")

RAG chain ready with OpenAI gpt-4o-mini


### 7.2 Test Full Pipeline

Run 4 representative questions through the complete pipeline (Semantic + Reranking) to verify end-to-end generation quality before formal evaluation.


In [28]:
test_questions = [
    "What was Apple's total net revenue in fiscal year 2024?",
    "What were Apple's earnings per share in 2024?",
    "How much did Apple spend on research and development?",
    "What are the main risk factors related to Apple's supply chain?",
]

for question in test_questions:
    print("=" * 80)
    print(f"Q: {question}")
    print("=" * 80)

    result = rag_answer(question, use_reranker=True)

    print(f"A: {result['answer']}")
    print(f"\n[Retrieved {len(result['sources'])} chunks]")
    print()

Q: What was Apple's total net revenue in fiscal year 2024?
A: Apple's total net revenue in fiscal year 2024 was $391,035 million.

[Retrieved 3 chunks]

Q: What were Apple's earnings per share in 2024?
A: Apple's earnings per share in 2024 were as follows: 

- Basic earnings per share: $6.11 
- Diluted earnings per share: $6.08

[Retrieved 3 chunks]

Q: How much did Apple spend on research and development?
A: Apple spent $31,370 million on research and development in 2024, $29,915 million in 2023, and $26,251 million in 2022.

[Retrieved 3 chunks]

Q: What are the main risk factors related to Apple's supply chain?
A: The main risk factors related to Apple's supply chain include:

1. **Natural Disasters**: Many of Apple's operations, retail stores, and facilities, as well as those of its suppliers and contract manufacturers, are located in areas prone to earthquakes and other natural disasters, which can disrupt manufacturing and delivery.

2. **Climate Change**: Global climate change i

## 8. RAGAS Evaluation

We formally evaluate three pipeline configurations using [RAGAS](https://docs.ragas.io):

| Experiment | Chunking | Reranking |
|---|---|---|
| **A** | Fixed-size | ❌ |
| **B** | Semantic | ❌ |
| **C** | Semantic | ✅ |

**Metrics measured:**
- **Faithfulness** — Does the answer stick to the retrieved context? (no hallucination)
- **Context Precision** — Are the retrieved chunks actually relevant to the question?
- **Context Recall** — Did retrieval find all the information needed to answer fully?

### 8.1 Evaluation Dataset

10 questions with ground truth answers manually verified from Apple's official 10-K filings.


In [29]:
eval_dataset = [
    {
        "question": "What was Apple's total net revenue in fiscal year 2024?",
        "ground_truth": "Apple's total net sales in fiscal year 2024 were $391,035 million."
    },
    {
        "question": "What were Apple's earnings per share in 2024?",
        "ground_truth": "Apple's basic EPS in 2024 was $6.11 and diluted EPS was $6.08."
    },
    {
        "question": "How much did Apple spend on research and development in 2024?",
        "ground_truth": "Apple spent $31,370 million on research and development in fiscal year 2024."
    },
    {
        "question": "What are the main risk factors related to Apple's supply chain?",
        "ground_truth": "Apple's supply chain risks include natural disasters, industrial accidents, public health issues like pandemics, cybersecurity attacks, labor disputes, fire and power shortages, and terrorist attacks that can interrupt manufacturing and delivery of products."
    },
    {
        "question": "What was Apple's net income in fiscal year 2024?",
        "ground_truth": "Apple's net income in fiscal year 2024 was $93,736 million."
    },
    {
        "question": "What was Apple's iPhone revenue in 2024?",
        "ground_truth": "Apple's iPhone revenue in fiscal year 2024 was $201,183 million."
    },
    {
        "question": "What was Apple's services revenue in 2024?",
        "ground_truth": "Apple's services revenue in fiscal year 2024 was $96,169 million, representing approximately 24.6% of total net sales."
    },
    {
        "question": "What was Apple's Mac revenue in fiscal year 2024?",
        "ground_truth": "Apple's Mac revenue in fiscal year 2024 was $29,984 million, a 2% increase from $29,357 million in 2023."
    },
    {
        "question": "What were Apple's Americas net sales in 2024?",
        "ground_truth": "Apple's Americas net sales in fiscal year 2024 were $167,045 million."
    },
    {
        "question": "What were the main factors affecting Apple's gross margin in 2024?",
        "ground_truth": "Apple's gross margin improved in 2024 primarily due to higher services revenue, which carries higher margins than hardware products, along with favorable product mix shifts."
    },
]

print(f"✅ Evaluation dataset ready: {len(eval_dataset)} questions")

✅ Evaluation dataset ready: 10 questions


### 8.2 Collect Results — 3 Experiments

Each question runs through all three pipeline configurations. Results include the generated answer and retrieved contexts for RAGAS to evaluate.


In [30]:
import time

results_fixed_only       = []   # Experiment A: Fixed, no rerank
results_semantic_only    = []   # Experiment B: Semantic, no rerank
results_semantic_rerank  = []   # Experiment C: Semantic + rerank

for i, item in enumerate(eval_dataset):
    q  = item["question"]
    gt = item["ground_truth"]
    print(f"[{i+1}/{len(eval_dataset)}] {q[:65]}...")

    # Experiment A: Fixed chunking, no reranking
    fixed_docs = retrieve_fixed(q, k=3)
    fixed_ctx  = format_context(fixed_docs)
    fixed_ans  = (prompt_template | llm | StrOutputParser()).invoke({"context": fixed_ctx, "question": q})
    results_fixed_only.append({
        "question":     q,
        "answer":       fixed_ans,
        "contexts":     [d.page_content for d in fixed_docs],
        "ground_truth": gt
    })

    # Experiment B: Semantic chunking, no reranking
    sem_docs = retrieve_no_rerank(q, k=3)
    sem_ctx  = format_context(sem_docs)
    sem_ans  = (prompt_template | llm | StrOutputParser()).invoke({"context": sem_ctx, "question": q})
    results_semantic_only.append({
        "question":     q,
        "answer":       sem_ans,
        "contexts":     [d.page_content for d in sem_docs],
        "ground_truth": gt
    })

    # Experiment C: Semantic chunking + reranking
    rerank_docs = retrieve_with_rerank(q, k=10, top_n=3)
    rerank_ctx  = format_context(rerank_docs)
    rerank_ans  = (prompt_template | llm | StrOutputParser()).invoke({"context": rerank_ctx, "question": q})
    results_semantic_rerank.append({
        "question":     q,
        "answer":       rerank_ans,
        "contexts":     [d.page_content for d in rerank_docs],
        "ground_truth": gt
    })

    time.sleep(1)  # small delay between questions

print(f"\n✅ Collected results for {len(eval_dataset)} questions across 3 experiments")

[1/10] What was Apple's total net revenue in fiscal year 2024?...
[2/10] What were Apple's earnings per share in 2024?...
[3/10] How much did Apple spend on research and development in 2024?...
[4/10] What are the main risk factors related to Apple's supply chain?...
[5/10] What was Apple's net income in fiscal year 2024?...
[6/10] What was Apple's iPhone revenue in 2024?...
[7/10] What was Apple's services revenue in 2024?...
[8/10] What was Apple's Mac revenue in fiscal year 2024?...
[9/10] What were Apple's Americas net sales in 2024?...
[10/10] What were the main factors affecting Apple's gross margin in 2024...

✅ Collected results for 10 questions across 3 experiments


### 8.3 Run RAGAS Metrics

RAGAS uses `gpt-4o-mini` as the judge LLM and OpenAI embeddings internally. Evaluates all 3 experiments across 10 questions.


In [37]:
!pip install -q --no-deps "langchain-community==0.4.1"

In [38]:
import importlib.metadata as md

print("RAGAS:", md.version("ragas"))
print("langchain-community:", md.version("langchain-community"))

RAGAS: 0.4.3
langchain-community: 0.4.1


In [39]:
import warnings
warnings.filterwarnings('ignore')

from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

ragas_llm        = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

metrics = [faithfulness, context_precision, context_recall]
for m in metrics:
    m.llm = ragas_llm
    if hasattr(m, "embeddings"):
        m.embeddings = ragas_embeddings

print("Evaluating Experiment A — Fixed chunking, no reranking...")
score_fixed = evaluate(Dataset.from_list(results_fixed_only), metrics=metrics)

print("\nEvaluating Experiment B — Semantic chunking, no reranking...")
score_semantic = evaluate(Dataset.from_list(results_semantic_only), metrics=metrics)

print("\nEvaluating Experiment C — Semantic chunking + reranking...")
score_rerank = evaluate(Dataset.from_list(results_semantic_rerank), metrics=metrics)

print("\n✅ All evaluations complete")

Evaluating Experiment A — Fixed chunking, no reranking...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


Evaluating Experiment B — Semantic chunking, no reranking...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


Evaluating Experiment C — Semantic chunking + reranking...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


✅ All evaluations complete


## 9. Results & Analysis


In [40]:
import numpy as np
import pandas as pd

def extract_score(score_obj, metric_name):
    val = score_obj[metric_name]
    if isinstance(val, list):
        clean = [v for v in val if v is not None and not np.isnan(v)]
        return round(sum(clean) / len(clean), 3) if clean else "N/A"
    return round(float(val), 3) if val is not None else "N/A"

metric_names = ["faithfulness", "context_precision", "context_recall"]
labels       = ["Faithfulness", "Context Precision", "Context Recall"]

comparison = pd.DataFrame({
    "Metric":                  labels,
    "A: Fixed (no rerank)":    [extract_score(score_fixed,    m) for m in metric_names],
    "B: Semantic (no rerank)": [extract_score(score_semantic, m) for m in metric_names],
    "C: Semantic + Reranking": [extract_score(score_rerank,   m) for m in metric_names],
})

print("=" * 70)
print("FINANCIAL RAG PIPELINE — EVALUATION REPORT")
print("=" * 70)
print(comparison.to_string(index=False))
print("\nScore range: 0.0 (worst) → 1.0 (best)")

print("""
METRIC EXPLANATIONS:
  Faithfulness      — Does the answer stick to the retrieved context?
                      High score = no hallucination, low = model making things up
  Context Precision — Are the retrieved chunks actually relevant to the question?
                      High score = clean retrieval, low = noisy irrelevant chunks
  Context Recall    — Did retrieval find all info needed to answer fully?
                      High score = complete coverage, low = missing key chunks

WHAT TO LOOK FOR:
  A vs B → Did semantic chunking improve over fixed-size?
  B vs C → Did reranking improve semantic chunking?
""")

# Quick winner summary
sem_cp = extract_score(score_semantic, "context_precision")
fix_cp = extract_score(score_fixed, "context_precision")
rer_cp = extract_score(score_rerank, "context_precision")

print("WINNER SUMMARY:")
if isinstance(sem_cp, float) and isinstance(fix_cp, float):
    winner = "Semantic" if sem_cp > fix_cp else "Fixed"
    print(f"  Chunking:  {winner} chunking wins on Context Precision ({sem_cp} vs {fix_cp})")
if isinstance(rer_cp, float) and isinstance(sem_cp, float):
    rerank_helps = "Yes" if rer_cp > sem_cp else "No"
    print(f"  Reranking: Does it help? {rerank_helps} ({rer_cp} vs {sem_cp})")

FINANCIAL RAG PIPELINE — EVALUATION REPORT
           Metric  A: Fixed (no rerank)  B: Semantic (no rerank)  C: Semantic + Reranking
     Faithfulness                   0.4                     0.90                    0.967
Context Precision                   0.0                     0.75                    1.000
   Context Recall                   0.0                     0.80                    1.000

Score range: 0.0 (worst) → 1.0 (best)

METRIC EXPLANATIONS:
  Faithfulness      — Does the answer stick to the retrieved context?
                      High score = no hallucination, low = model making things up
  Context Precision — Are the retrieved chunks actually relevant to the question?
                      High score = clean retrieval, low = noisy irrelevant chunks
  Context Recall    — Did retrieval find all info needed to answer fully?
                      High score = complete coverage, low = missing key chunks

WHAT TO LOOK FOR:
  A vs B → Did semantic chunking improve over fi